In [1]:
import arcpy
import os
arcpy.env.overwriteOutput = True

In [2]:
model_folder = r"C:/Users/Chels/OneDrive - University of Illinois - Urbana/Ch6_CASC_Project/Dual-Risk-Repo/Carbon_Storage_Estimation/NWCA_Model"

In [3]:
tc_points = os.path.normpath(os.path.join(model_folder,"Step7_IL_NWCA_TC_Points_NAD1983.shp"))

In [4]:
raster_paths = [
    "F:/databases/climate/prism_ppt_us_30s_2020_avg_30y/ppt_avg_30y_IL_LambertNAD1983.tif",
    "F:/databases/climate/prism_tmean_us_30s_2020_avg_30y/tmean_avg_30y_IL_LambertNAD1983.tif",
    "F:/wim_rasters/dem/IL_DEM_3m_LambertNAD1983.tif",
    "F:/wim_rasters/slope/Slope_3m_LambertNAD1983.tif",
    "F:/wim_rasters/aspect/Aspect_3m_LambertNAD1983.tif",
    "F:/wim_rasters/twi/TWI_IL_Final_LambertNAD1983_Mosaic.tif",
    "F:/wim_rasters/ndwi/NDWI_IL_LambertNAD1983.tif",
    "F:/wim_rasters/ndvi/NDVI_IL_Final_LambertNAD1983.tif",
]

In [5]:
raster_labels = [
    "MAP",
    "MAT",
    "Elevation",
    "Slope",
    "Aspect",
    "TWI",
    "NDWI",
    "NDVI",
]
n_r = len(raster_labels)

In [6]:
# build raster list object for ExtractMultiValuesToPoints
raster_list = [[raster_paths[i], raster_labels[i]] for i in range(n_r)]
raster_list

[['F:/databases/climate/prism_ppt_us_30s_2020_avg_30y/ppt_avg_30y_IL_LambertNAD1983.tif',
  'MAP'],
 ['F:/databases/climate/prism_tmean_us_30s_2020_avg_30y/tmean_avg_30y_IL_LambertNAD1983.tif',
  'MAT'],
 ['F:/wim_rasters/dem/IL_DEM_3m_LambertNAD1983.tif', 'Elevation'],
 ['F:/wim_rasters/slope/Slope_3m_LambertNAD1983.tif', 'Slope'],
 ['F:/wim_rasters/aspect/Aspect_3m_LambertNAD1983.tif', 'Aspect'],
 ['F:/wim_rasters/twi/TWI_IL_Final_LambertNAD1983_Mosaic.tif', 'TWI'],
 ['F:/wim_rasters/ndwi/NDWI_IL_LambertNAD1983.tif', 'NDWI'],
 ['F:/wim_rasters/ndvi/NDVI_IL_Final_LambertNAD1983.tif', 'NDVI']]

In [7]:
# project points to lambert
tc_points_project = os.path.normpath(os.path.join(model_folder,"Step7_IL_NWCA_TC_Points_Lambert.shp"))

In [8]:
# project to Lambert conformal conic 
arcpy.management.Project(
    in_dataset=tc_points,
    out_dataset=tc_points_project,
    out_coor_system=arcpy.SpatialReference(102009)  #lambert conformal conic
);

In [9]:
arcpy.management.CopyFeatures(tc_points_project, r"memory\scratch_points");

In [10]:
arcpy.sa.ExtractMultiValuesToPoints(
    in_point_features=r"memory\scratch_points",
    in_rasters=raster_list,
    bilinear_interpolate_values="NONE"
);

In [11]:
arcpy.conversion.ExportFeatures(r"memory\scratch_points",tc_points_project);

In [12]:
arcpy.management.Delete(r"memory\scratch_points");
print("All raster values extracted to permit point attribute table.")

All raster values extracted to permit point attribute table.


In [13]:
arcpy.conversion.TableToTable(tc_points_project, model_folder, "Step8_IL_NWCA_TC_Points_Attributes.csv");